# 04 Beauty

## Step 1: Save a copy to your Google Drive
Go to **File → Save a copy in Drive**, then remove `Copy of` from the filename.

## Step 2: Mount Google Drive and download the image
Run the cell below to mount Google Drive and download the input image.

## Step 3: Write the file
Run the cell to save the Python script to your project folder.

## Step 4: Run and check output
Run the saved script and check the output.

In [ ]:
# Step 2: Mount Google Drive, set up project folder, and download image
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/Practical-IP/04"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"

IMAGE_URL = 'https://raw.githubusercontent.com/mastnk/practical-ip/main/04/Face.png'
!wget -nc -q "{IMAGE_URL}" -O Face.png
!ls *.py *.png

In [ ]:
# Step 3: Run this cell to save the Python file
%%writefile Beauty.py

import sys

import cv2 # it is necessary to use cv2 library
import numpy as np

def hsvwithinrange( hsv ):
    h = hsv[:,:,0]

    while( np.count_nonzero( h > 360 ) > 0 ):
        h[h>360] -= 360

    while( np.count_nonzero( h < 0 ) > 0 ):
        h[h<0] += 360

    hsv[:,:,0] = h

    hsv[:,:,1:] = hsv[:,:,1:].clip(0,1)

    return hsv

def main( input_filename, output_filename ):
    img = cv2.imread( input_filename )  # load image from input_filename
    img = img.astype('float32')/255.0   # single [0,1]

    ############ Edit here ############

    sigma = 2.5
    smooth = 0.025
    enhance = 1.0
    iteration = 1

    V_gamma = 1.0
    S_gain = 1.0

    ############ Edit here ############

    ksize = int(np.ceil(sigma*3))*2+1

    (height, width, channel) = img.shape
    for c in range(channel):
        for i in range(iteration):
            m1 = img[:,:,c]

            Lm1 = cv2.GaussianBlur( m1, ksize=(ksize, ksize), sigmaX=sigma, borderType=cv2.BORDER_REPLICATE )
            m2 = m1 - Lm1
            m2 = m2 * m2;
            Lm2 = cv2.GaussianBlur( m2, ksize=(ksize, ksize), sigmaX=sigma, borderType=cv2.BORDER_REPLICATE )

            img[:,:,c] = (m1 - Lm1) * Lm2 / ( Lm2 + smooth*smooth) * enhance + Lm1


    #If you want to use hsv, uncomment the next line
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    hsv[:,:,2] = np.power( hsv[:,:,2].clip(0,1), V_gamma )
    hsv[:,:,1] = hsv[:,:,1] * S_gain

    #If you want to use hsv, uncomment the next line
    img = cv2.cvtColor(hsvwithinrange(hsv), cv2.COLOR_HSV2BGR)

    img = (img * 255).clip( 0, 255 ).astype('uint8') # uint8 [0,255]
    cv2.imwrite( output_filename, img ) # save image to output_filename

if( __name__ == '__main__' ):
    if( len(sys.argv) >= 3 ):
        main( sys.argv[1], sys.argv[2] )
    else:
        print( 'usage: python '+sys.argv[0]+' input_filenname output_filename' )

In [ ]:
# Step 4: Run the saved file
%cd "{PROJECT_PATH}"
!python Beauty.py Face.png Beauty.png
!ls *.png

In [ ]:
# Step 5: Display input and output images
from google.colab.patches import cv2_imshow
import cv2

print('Face')
cv2_imshow( cv2.imread('Face.png') )
print('Beauty')
cv2_imshow( cv2.imread('Beauty.png') )